# Práctica de Deep Learning para Finanzas · Avance 4 (capítulo 10)
**Ciencia de Datos Aplicada a la Economía** · Prof. Rubén Miguel Rodríguez Rangel · Luis Enrique López Salazar

Tema: aprendizaje por refuerzo. Se diseña el **entorno** del futuro agente (estado, acciones, restricciones y recompensa) y se
calcula una transición a mano. Todavía no se entrena ningún agente.

Entregable (protocolo §5): especificación del entorno y transición calculada. El entorno usa solo fechas de VA; TE se descarta
en la celda 2.

**Cómo correrlo:** en Colab o en cualquier Python 3 (no necesita GPU). *Ejecutar todas*; la celda 2 pide subir `avance4_insumos.zip`.

## 1. Configuración e insumos de los Avances 1 y 3

In [1]:
import json, os, platform, shutil, sys
import numpy as np, pandas as pd

VENTANA = 20
CAPITAL, COSTO = 100_000, 0.001          # protocolo §9: 100,000 USDT y 0.10 % por compra y por venta
LAGS = [f"r_lag{i}" for i in range(VENTANA)]   # r_lag0 = rendimiento al cierre de t; r_lag19 = el de hace 19 días
os.makedirs("outputs", exist_ok=True)
INSUMOS = ["data/processed/ventanas_btcusdt_1d.csv", "data/processed/btcusdt_1d_auditado.csv", "outputs/particiones_avance1.csv",
           "outputs/estimados_va_avance3.csv", "outputs/regla_avance3.json", "outputs/simulacion_va_avance3.csv"]
if "google.colab" in sys.modules and not all(map(os.path.exists, INSUMOS)):
    from google.colab import files
    print("Sube avance4_insumos.zip")
    shutil.unpack_archive(next(iter(files.upload())), ".")
faltan = [f for f in INSUMOS if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan insumos: {faltan}. Detengo el flujo.")
REGLA = json.load(open(INSUMOS[4]))
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})
print("Pronóstico del estado:", REGLA["modelo"], "| regla del Avance 3:", REGLA["regla"])

{'python': '3.12.3', 'numpy': '2.5.5', 'pandas': '3.1.1'}
Pronóstico del estado: Regresión lineal (20 rezagos) | regla del Avance 3: Al cierre de t: comprar (100 %) si el rendimiento estimado para t+1 > 0.05%; si no, efectivo. Ejecutar en la apertura de t+1.


## 2. Datos de VA y reserva de TE
Cada día de VA aporta: los 20 rendimientos conocidos al cierre de *t*, el pronóstico ŷ que el modelo del Avance 3 emitió ese cierre
y los precios de *t*+1 para ejecutar y valorar. Se descartan las filas de TE y los precios desde su primera fecha.

In [2]:
ref = pd.read_csv(INSUMOS[2]).set_index("conjunto")
INICIO_TE = pd.Timestamp(ref.loc["TE", "primera_fecha_objetivo"])
datos = pd.read_csv(INSUMOS[0], parse_dates=["fecha_emision", "fecha_objetivo"])
VA = datos[datos.conjunto == "VA"].sort_values("fecha_emision").reset_index(drop=True)
est = pd.read_csv(INSUMOS[3], parse_dates=["fecha_emision", "fecha_objetivo"])
VA = VA.merge(est, on=["fecha_emision", "fecha_objetivo"], validate="1:1")
precios = pd.read_csv(INSUMOS[1], parse_dates=["fecha"]).set_index("fecha")
precios = precios[precios.index < INICIO_TE]
assert len(VA) == ref.loc["VA", "ejemplos"] and VA.fecha_objetivo.max() < INICIO_TE
assert (VA.fecha_emision.iloc[1:].values == VA.fecha_objetivo.iloc[:-1].values).all(), "VA no es una serie diaria continua"
# r_lag0 es el rendimiento del cierre de t: se recalcula con precios hasta t
r0 = precios.close.loc[VA.fecha_emision].to_numpy() / precios.close.shift(1).loc[VA.fecha_emision].to_numpy() - 1
assert np.allclose(VA.r_lag0, r0), "r_lag0 no corresponde al cierre de t"
print(f"VA: {len(VA)} días, emisión {VA.fecha_emision.min().date()} … {VA.fecha_emision.max().date()} | TE descartado desde {INICIO_TE.date()}")

VA: 442 días, emisión 2024-06-01 … 2025-08-16 | TE descartado desde 2025-08-18


## 3. Especificación del entorno
| Elemento | Definición | Cuándo se conoce |
|---|---|---|
| **Paso** | Un día de VA. Decisión al cierre de *t*, ejecución en la apertura de *t*+1, valoración al cierre de *t*+1 | — |
| **Estado** | 20 rendimientos diarios (r<sub>t</sub> … r<sub>t−19</sub>), pronóstico ŷ<sub>t+1</sub> del modelo del Avance 3, volatilidad de esos 20 rendimientos (desviación estándar), efectivo y posición (unidades de BTC) | Todo al cierre de *t*. No incluye la apertura ni el cierre de *t*+1 ni el rendimiento real *y* |
| **Acciones** | **comprar** (de efectivo a 100 % en BTC), **mantener** (dejar todo como está), **vender** (de 100 % en BTC a efectivo) | — |
| **Mantener ≠ efectivo** | *Mantener* con posición abierta = seguir expuesto al precio. *Mantener* sin posición = **permanecer en efectivo**: el patrimonio no cambia. Mismo nombre, distinto riesgo; por eso el estado incluye la posición | — |
| **Restricciones** | Sin apalancamiento ni posiciones negativas: *comprar* solo sin posición (la compra usa el efectivo disponible e incluye el costo); *vender* solo con posición. Una acción fuera de esto **se rechaza** (error), no se ejecuta | — |
| **Costos** | 0.10 % sobre el monto negociado, en cada compra y cada venta; *mantener* no paga | En la ejecución |
| **Recompensa** | r<sub>t+1</sub> = patrimonio(cierre *t*+1) − patrimonio(cierre *t*), con el costo ya descontado | Al cierre de *t*+1, un día después de decidir |
| **Fin del episodio** | Último día de VA. Si queda posición, se liquida al último cierre cobrando la venta (regla 6 del §9) y entra en la última recompensa | — |

Patrimonio = efectivo + unidades × cierre. Con esta recompensa, la suma de recompensas del episodio es exactamente patrimonio final − capital.

## 4. El entorno en código

In [3]:
class Entorno:
    """Un paso por día de VA. reiniciar() → estado; paso(acción) → (nuevo estado, recompensa, fin, detalle)."""
    ACCIONES = ("comprar", "mantener", "vender")

    def __init__(self, dias, costo=COSTO):
        self.dias, self.costo = dias.reset_index(drop=True), costo

    def reiniciar(self):
        self.i, self.efectivo, self.unidades = 0, float(CAPITAL), 0.0
        return self.estado()

    def patrimonio(self, fecha):
        return self.efectivo + self.unidades * precios.loc[fecha, "close"]

    def estado(self):
        d = self.dias.iloc[self.i]
        r = d[LAGS].to_numpy(float)
        return {"cierre_t": d.fecha_emision, "rendimientos_20": r, "pronostico": float(d.estimado),
                "volatilidad_20": float(r.std(ddof=1)), "efectivo": self.efectivo, "posicion": self.unidades}

    def validas(self):
        return ("comprar", "mantener") if self.unidades == 0 else ("mantener", "vender")

    def paso(self, accion):
        if accion not in self.validas():
            raise ValueError(f"acción rechazada: {accion} con posición {self.unidades:.6f} BTC y efectivo {self.efectivo:,.2f}")
        d = self.dias.iloc[self.i]
        t, t1 = d.fecha_emision, d.fecha_objetivo
        antes, apertura, costo = self.patrimonio(t), precios.loc[t1, "open"], 0.0
        if accion == "comprar":     # todo el efectivo, con el costo incluido: unidades × apertura × (1 + costo) = efectivo
            self.unidades = self.efectivo / (apertura * (1 + self.costo))
            costo, self.efectivo = self.unidades * apertura * self.costo, 0.0
        elif accion == "vender":
            bruto = self.unidades * apertura
            costo, self.efectivo, self.unidades = bruto * self.costo, bruto * (1 - self.costo), 0.0
        self.i += 1
        fin = self.i == len(self.dias)
        if fin and self.unidades > 0:   # liquidación al último cierre
            bruto = self.unidades * precios.loc[t1, "close"]
            costo, self.efectivo, self.unidades = costo + bruto * self.costo, bruto * (1 - self.costo), 0.0
        assert self.efectivo >= 0 and self.unidades >= 0, "apalancamiento o posición negativa"
        recompensa = self.patrimonio(t1) - antes
        detalle = {"cierre_t": t.date(), "ejecucion": t1.date(), "accion": accion, "apertura_t1": apertura,
                   "costo": costo, "posicion_t1": self.unidades, "patrimonio_t1": self.patrimonio(t1)}
        return (None if fin else self.estado()), recompensa, fin, detalle

entorno = Entorno(VA)
e = entorno.reiniciar()
assert set(e) == {"cierre_t", "rendimientos_20", "pronostico", "volatilidad_20", "efectivo", "posicion"}, "el estado tiene algo de más"
print("Estado inicial:", {k: (v.round(4) if isinstance(v, np.ndarray) else v) for k, v in e.items()})

Estado inicial: {'cierre_t': Timestamp('2024-06-01 00:00:00'), 'rendimientos_20': array([ 0.0032, -0.0118,  0.0103, -0.0109, -0.0149,  0.0136, -0.0113,
        0.0108,  0.0084, -0.0171, -0.014 , -0.0182,  0.078 , -0.0096,
       -0.0016,  0.0274, -0.0147,  0.0752, -0.0216,  0.0237]), 'pronostico': 0.0002572605337957973, 'volatilidad_20': 0.028283716662705426, 'efectivo': 100000.0, 'posicion': 0.0}


## 5. Transición calculada a mano
Primer día de VA, sin posición, acción **comprar** (elegida para el ejemplo; no la decide una política).
Los números se calculan con las fórmulas escritas, sin la clase, y después se comparan con `Entorno.paso`.

In [4]:
d0 = VA.iloc[0]
t, t1 = d0.fecha_emision, d0.fecha_objetivo
P_ap, P_ci = precios.loc[t1, "open"], precios.loc[t1, "close"]
unidades = CAPITAL / (P_ap * (1 + COSTO))
costo = unidades * P_ap * COSTO
efectivo_nuevo = CAPITAL - unidades * P_ap - costo
patrimonio_nuevo = efectivo_nuevo + unidades * P_ci
recompensa = patrimonio_nuevo - CAPITAL
r20 = d0[LAGS].to_numpy(float)
mano = pd.DataFrame([
    ["estado (cierre de t)", str(t.date()), f"efectivo {CAPITAL:,.2f} · posición 0 BTC · ŷ = {d0.estimado:+.4%} · volatilidad 20 = {r20.std(ddof=1):.4%} · r_t = {r20[0]:+.4%}"],
    ["acción", str(t.date()), "comprar"],
    ["ejecución (apertura de t+1)", str(t1.date()), f"precio {P_ap:,.2f} · unidades = {CAPITAL:,.0f} / ({P_ap:,.2f} × 1.001) = {unidades:.6f} BTC · costo = {costo:,.2f}"],
    ["valoración (cierre de t+1)", str(t1.date()), f"precio {P_ci:,.2f} · patrimonio = {efectivo_nuevo:,.2f} + {unidades:.6f} × {P_ci:,.2f} = {patrimonio_nuevo:,.2f}"],
    ["recompensa", str(t1.date()), f"{patrimonio_nuevo:,.2f} − {CAPITAL:,.2f} = {recompensa:+,.2f} USDT"],
    ["nuevo estado (cierre de t+1)", str(t1.date()), f"efectivo {efectivo_nuevo:,.2f} · posición {unidades:.6f} BTC · ŷ = {VA.estimado.iloc[1]:+.4%} · r_t+1 = {VA.r_lag0.iloc[1]:+.4%}"],
], columns=["paso", "fecha", "cálculo"])
mano.to_csv("outputs/transicion_manual.csv", index=False)

ent = Entorno(VA); ent.reiniciar()
e1, r_clase, fin, det = ent.paso("comprar")
assert abs(r_clase - recompensa) < 1e-6 and abs(e1["posicion"] - unidades) < 1e-12 and abs(e1["efectivo"] - efectivo_nuevo) < 1e-6
assert abs(det["costo"] - costo) < 1e-9 and e1["cierre_t"] == t1
print(f"La clase da la misma transición: recompensa {r_clase:+,.2f} USDT, {e1['posicion']:.6f} BTC.")
pd.set_option("display.max_colwidth", None)
mano

La clase da la misma transición: recompensa -101.68 USDT, 1.474174 BTC.


,paso,fecha,cálculo
0,estado (cierre de t),2024-06-01,"efectivo 100,000.00 · posición 0 BTC · ŷ = +0.0257% · volatilidad 20 = 2.8284% · r_t = +0.3240%"
1,acción,2024-06-01,comprar
2,ejecución (apertura de t+1),2024-06-02,"precio 67,766.84 · unidades = 100,000 / (67,766.84 × 1.001) = 1.474174 BTC · costo = 99.90"
3,valoración (cierre de t+1),2024-06-02,"precio 67,765.63 · patrimonio = 0.00 + 1.474174 × 67,765.63 = 99,898.32"
4,recompensa,2024-06-02,"99,898.32 − 100,000.00 = -101.68 USDT"
5,nuevo estado (cierre de t+1),2024-06-02,efectivo 0.00 · posición 1.474174 BTC · ŷ = +0.3589% · r_t+1 = -0.0018%


## 6. El entorno reproduce el simulador del Avance 3
Se recorre VA con la regla seleccionada en el Avance 3, traducida a acciones (señal comprar sin posición → *comprar*;
señal efectivo con posición → *vender*; lo demás → *mantener*). El patrimonio de cada cierre debe coincidir con `simulacion_va.csv`,
y la suma de recompensas con patrimonio final − capital.

In [5]:
def episodio(politica, costo=COSTO):
    ent = Entorno(VA, costo)
    e, fin, filas = ent.reiniciar(), False, []
    while not fin:
        accion = politica(e)
        e, r, fin, det = ent.paso(accion)
        filas.append({**det, "recompensa": r})
    return pd.DataFrame(filas)

DELTA = REGLA["umbral"]
def regla_av3(e):
    quiere = e["pronostico"] > DELTA
    if quiere and e["posicion"] == 0: return "comprar"
    if not quiere and e["posicion"] > 0: return "vender"
    return "mantener"

ep = episodio(regla_av3)
sim3 = pd.read_csv(INSUMOS[5])
assert np.allclose(ep.patrimonio_t1.to_numpy(), sim3.patrimonio.to_numpy(), rtol=0, atol=1e-6), "el entorno no reproduce el Avance 3"
assert abs(ep.recompensa.sum() - (ep.patrimonio_t1.iloc[-1] - CAPITAL)) < 1e-6
ep.to_csv("outputs/episodio_regla_va.csv", index=False)
print(f"{len(ep)} pasos | patrimonio final {ep.patrimonio_t1.iloc[-1]:,.2f} = Avance 3 | suma de recompensas {ep.recompensa.sum():+,.2f} | "
      f"costos {ep.costo.sum():,.2f} | acciones: {ep.accion.value_counts().to_dict()}")

442 pasos | patrimonio final 132,538.32 = Avance 3 | suma de recompensas +32,538.32 | costos 20,860.26 | acciones: {'mantener': 241, 'comprar': 101, 'vender': 100}


## 7. Pronóstico, señal y política
- **Pronóstico:** un número, ŷ<sub>t+1</sub>, el rendimiento que el modelo estima para mañana. No dice qué hacer.
- **Señal:** una regla fija escrita por nosotros que convierte el pronóstico en una decisión, p. ej. «comprar si ŷ > 0.05 %» (Avance 3).
  No mira la posición, los costos ni el futuro más allá de un día.
- **Política aprendida:** una función π(estado) → acción que el agente aprende para **maximizar la suma de recompensas futuras**.
  Ve el pronóstico como *una variable más* del estado (junto a la volatilidad, el efectivo y la posición) y puede aprender, por ejemplo,
  a no vender por una caída pequeña porque la recompra cuesta 0.20 % ida y vuelta.

## 8. Exploración, memoria de experiencias y red objetivo (DQN / DDQN)
DQN aproxima Q(s, a), la recompensa acumulada esperada si en el estado *s* se toma *a* y después se sigue la política, con una red.
- **Exploración (ε-greedy):** con probabilidad ε el agente elige una acción válida al azar; si no, la de mayor Q. ε empieza cerca de 1
  y baja: sin explorar, el agente nunca prueba acciones que parecen peores al principio y puede quedarse en «siempre efectivo».
- **Memoria de experiencias (*replay buffer*):** se guardan las transiciones (s, a, r, s′, fin) y se entrena con lotes al azar.
  Rompe la correlación de días consecutivos y reutiliza cada experiencia varias veces.
- **Red objetivo:** una copia de la red, congelada y actualizada cada *N* pasos, calcula el objetivo r + γ·max Q<sub>obj</sub>(s′, a′).
  Si el objetivo saliera de la misma red que se entrena, perseguiría un blanco que se mueve con ella y la estimación se vuelve inestable.
- **DDQN:** DQN tiende a sobreestimar Q porque el mismo max elige y evalúa la acción. DDQN **elige** a′ con la red en línea y la
  **evalúa** con la red objetivo: r + γ·Q<sub>obj</sub>(s′, argmax<sub>a′</sub> Q(s′, a′)).
- Las acciones inválidas (*comprar* con posición, *vender* sin ella) se **enmascaran** al elegir y en el max del objetivo.

## 9. Seguimiento: ¿el agente podría aprovechar un error del simulador?
Sí: un agente que maximiza recompensa explota cualquier hueco, porque para él es recompensa como cualquier otra.
Se prueban cinco trampas. El entorno debe rechazar las dos primeras; las otras muestran cuánto ganaría con un simulador mal hecho.

In [6]:
trampas = []
def rechaza(accion, preparar):
    ent = Entorno(VA); ent.reiniciar(); preparar(ent)
    try:
        ent.paso(accion); return False, "aceptada"
    except ValueError as err:
        return True, str(err)

ok, msg = rechaza("comprar", lambda ent: ent.paso("comprar"))
trampas.append({"trampa": "comprar con la posición ya abierta (más efectivo del que hay: apalancamiento)", "defensa": "rechazo", "pasa": ok, "resultado": msg})
ok, msg = rechaza("vender", lambda ent: None)
trampas.append({"trampa": "vender sin posición (venta en corto)", "defensa": "rechazo", "pasa": ok, "resultado": msg})

e = Entorno(VA).reiniciar()
ok = not any(k in e for k in ("y", "fecha_objetivo", "apertura_t1")) and e["cierre_t"] == VA.fecha_emision.iloc[0]
trampas.append({"trampa": "ver el rendimiento real de t+1 en el estado", "defensa": "el estado no lo incluye", "pasa": ok,
                "resultado": "estado con datos hasta el cierre de t"})

def espia(e):   # política imposible: conoce y = cierre(t+1) / cierre(t) − 1
    y = VA.set_index("fecha_emision").y.loc[e["cierre_t"]]
    return ("comprar" if e["posicion"] == 0 else "mantener") if y > 0 else ("vender" if e["posicion"] > 0 else "mantener")
g_espia, g_regla = episodio(espia).recompensa.sum(), ep.recompensa.sum()
trampas.append({"trampa": "información futura (si el estado trajera y)", "defensa": "medida", "pasa": True,
                "resultado": f"ganaría {g_espia:+,.0f} USDT frente a {g_regla:+,.0f} de la regla: una recompensa así delata fuga del futuro"})

def cambiar(e):  # alterna todos los días: sin costos parece inofensivo
    return "vender" if e["posicion"] > 0 else "comprar"
c0, c1 = episodio(cambiar, costo=0.0), episodio(cambiar)
ok = c1.costo.sum() > 0 and c0.recompensa.sum() - c1.recompensa.sum() > 0
trampas.append({"trampa": "operar a diario si el simulador no cobrara costos", "defensa": "costo en cada compra y venta", "pasa": ok,
                "resultado": f"{len(c1)} operaciones: sin costos {c0.recompensa.sum():+,.0f}, con costos {c1.recompensa.sum():+,.0f} USDT "
                             f"({c1.costo.sum():,.0f} en costos)"})
trampas = pd.DataFrame(trampas)
trampas.to_csv("outputs/trampas.csv", index=False)
trampas

,trampa,defensa,pasa,resultado
0,comprar con la posición ya abierta (más efectivo del que hay: apalancamiento),rechazo,True,acción rechazada: comprar con posición 1.474174 BTC y efectivo 0.00
1,vender sin posición (venta en corto),rechazo,True,"acción rechazada: vender con posición 0.000000 BTC y efectivo 100,000.00"
2,ver el rendimiento real de t+1 en el estado,el estado no lo incluye,True,estado con datos hasta el cierre de t
3,información futura (si el estado trajera y),medida,True,"ganaría +4,560,562 USDT frente a +32,538 de la regla: una recompensa así delata fuga del futuro"
4,operar a diario si el simulador no cobrara costos,costo en cada compra y venta,True,"442 operaciones: sin costos +53,346, con costos -1,437 USDT (44,505 en costos)"


**Lectura.** El entorno rechaza el apalancamiento y los cortos, el estado no trae nada posterior al cierre de *t* y cada operación paga
su costo. Quedan dos riesgos que no se resuelven con código y se revisan antes de entrenar:
(1) que el pronóstico ŷ de VA venga de un modelo ajustado con VA (aquí viene de un modelo entrenado solo con TR, Avance 3);
(2) que la política se elija con muchos intentos sobre VA, que es otra forma de sobreajuste. TE sigue sin usarse.

## 10. Comprobación final

In [7]:
ent = Entorno(VA); ent.reiniciar(); sin_pos = ent.validas(); ent.paso("comprar"); con_pos = ent.validas()
chk = {
    "estado con 20 rendimientos, pronóstico, volatilidad, efectivo y posición": len(entorno.reiniciar()["rendimientos_20"]) == 20,
    "mantener válida con y sin posición; comprar solo sin ella, vender solo con ella": (sin_pos, con_pos) == (("comprar", "mantener"), ("mantener", "vender")),
    "transición manual = clase": abs(r_clase - recompensa) < 1e-6,
    "el entorno reproduce el simulador del Avance 3": np.allclose(ep.patrimonio_t1, sim3.patrimonio, atol=1e-6),
    "suma de recompensas = patrimonio final − capital": abs(ep.recompensa.sum() - (ep.patrimonio_t1.iloc[-1] - CAPITAL)) < 1e-6,
    "costos contabilizados": ep.costo.sum() > 0,
    "trampas: rechazos y defensas": bool(trampas.pasa.all()),
    "posición final liquidada": ep.posicion_t1.iloc[-1] == 0,
    "TE sin intervenir": VA.fecha_objetivo.max() < INICIO_TE and precios.index.max() < INICIO_TE,
}
for k, v in chk.items():
    print(("OK   " if v else "FALLA ") + k)
json.dump({"modelo_pronostico": REGLA["modelo"], "umbral_regla": DELTA, "recompensa_transicion_manual": recompensa,
           "patrimonio_final_regla": float(ep.patrimonio_t1.iloc[-1]), "comprobaciones": {k: bool(v) for k, v in chk.items()}},
          open("outputs/resumen.json", "w"), ensure_ascii=False, indent=1)

OK   estado con 20 rendimientos, pronóstico, volatilidad, efectivo y posición
OK   mantener válida con y sin posición; comprar solo sin ella, vender solo con ella
OK   transición manual = clase
OK   el entorno reproduce el simulador del Avance 3
OK   suma de recompensas = patrimonio final − capital
OK   costos contabilizados
OK   trampas: rechazos y defensas
OK   posición final liquidada
OK   TE sin intervenir


**Qué sigue (Avance 5, capítulo 11):** dos indicadores (SMA20 como razón y volatilidad de 20 días), el modelo con y sin ellos,
la única evaluación en TE y el pronóstico del siguiente periodo.